# Fetch OpenSky Data for Pushback Dataset Generation

This notebook fetches the OpenSky data required to create the automatically labeled training and validation datasets.

It performs two main tasks:

1. Fetch **history data** for all flights associated with the selected airport and time period.
2. Fetch and combine **raw ADS-B data** from the relevant OpenSky tables in hourly intervals.

Airport, season, start time, and stop time are defined once in the configuration section and are shared by both data-fetching steps.

## 1. Imports

In [ ]:
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd

from pyopensky.schema import (
    IdentificationData4,
    PositionData4,
    RollcallRepliesData4,
    VelocityData4,
)

from traffic.core import Traffic
from traffic.data import opensky, airports

from cartes.osm.overpass.core import Relation
from cartes.osm.overpass import Overpass

## 2. Configuration

Define the airport, season, and time period once here. These values are shared by the history and raw-data queries.

Data are stored relative to the repository root:

- `data/train_validation/01_source/raw/<AIRPORT>/<SEASON>/`
- `data/train_validation/01_source/history/<AIRPORT>/<SEASON>/`

The required directories are created automatically.


In [ ]:
airport_str = "LSZH"
season = "Spring"  # Spring, Summer, Autumn, or Winter

start = "2026-04-20 00:00"
stop = "2026-04-27 00:00"

valid_seasons = {"Spring", "Summer", "Autumn", "Winter"}
if season not in valid_seasons:
    raise ValueError(
        f"Invalid season '{season}'. Expected one of: {sorted(valid_seasons)}"
    )

# Project and data directories
PROJECT_ROOT = Path.home() / "Projects" / "OSN_pushback_detection"
DATA_ROOT = PROJECT_ROOT / "data"

source_root = DATA_ROOT / "train_validation" / "01_source"

raw_dir = source_root / "raw" / airport_str / season
history_dir = source_root / "history" / airport_str / season

raw_dir.mkdir(parents=True, exist_ok=True)
history_dir.mkdir(parents=True, exist_ok=True)

# Geographic extent of the selected airport
bounds = airports[airport_str].shape.convex_hull.buffer(0.1)
min_lon, min_lat, max_lon, max_lat = bounds.bounds


class Site(Relation):
    pass


Overpass.endpoint = "https://overpass.kumi.systems/api/interpreter"

## 3. Fetch History Data

The selected time period is divided into 24-hour intervals. For each interval, the flight list associated with the selected airport is retrieved first. The corresponding trajectory history is then fetched within the buffered airport extent.

All returned trajectories are combined and stored as one Parquet file.

In [ ]:
timestamps = pd.date_range(start, stop, freq="24h")

data = []

for t1, t2 in zip(timestamps[:-1], timestamps[1:]):
    print(t1, t2)

    flight_list = opensky.flightlist(
        start=t1,
        stop=t2,
        airport=airport_str,
    )

    if flight_list is None:
        continue

    icao24 = flight_list.icao24

    tmp = opensky.history(
        start=str(t1),
        stop=str(t2),
        bounds=bounds,
        icao24=icao24,
        return_flight=True,
    )

    if tmp is not None:
        data.append(tmp.data)

trajs = Traffic(pd.concat(data)) if data else Traffic(pd.DataFrame())

history_start = pd.Timestamp(start).strftime("%Y-%m-%d")
history_stop = (pd.Timestamp(stop) - pd.Timedelta(days=1)).strftime("%Y-%m-%d")

history_path = (
    history_dir / f"history_{history_start}_{history_stop}.parquet"
)

trajs.to_parquet(history_path)

## 4. Define Hourly Raw-Data Query Windows

Raw ADS-B data are fetched in one-hour intervals. Existing hourly Parquet files are skipped.

In [ ]:
start_dt = datetime.strptime(start, "%Y-%m-%d %H:%M")
stop_dt = datetime.strptime(stop, "%Y-%m-%d %H:%M")

one_hour = timedelta(hours=1)

windows = []
current = start_dt

while current < stop_dt:
    windows.append((current, current + one_hour))
    current += one_hour

## 5. Fetch and Combine Raw ADS-B Data

For each hourly interval:

1. Identify aircraft associated with the selected airport area.
2. Fetch position data from `PositionData4`.
3. Fetch velocity data from `VelocityData4`.
4. Derive the geometric-minus-barometric altitude difference.
5. Merge position and velocity information.
6. Fetch and merge squawk information from `RollcallRepliesData4`.
7. Fetch and merge callsign information from `IdentificationData4`.
8. Restrict each aircraft sequence to the interval containing valid position information.
9. Save the resulting hourly trajectories as a Parquet file.

The geographic filter is derived directly from the buffered extent of the selected airport.

In [ ]:
for hour in windows:
    starttime = hour[0].strftime("%Y-%m-%d %H:%M")
    stoptime = hour[1].strftime("%Y-%m-%d %H:%M")
    print(f"start: {starttime}, stop: {stoptime}")

    file_path = raw_dir / (
        starttime[0:10] + "_" + starttime[11:13] + "-" + stoptime[-5:-3] + ".parquet"
    )

    if not file_path.exists():
        try:
            # Identify aircraft associated with the selected airport area.
            icao24s = opensky.rawdata(
                starttime,
                stoptime,
                airport=airport_str,
                Table=PositionData4,
            ).data.icao24.unique()

            # -------------------------------------------------------------------------
            # Position data
            # -------------------------------------------------------------------------

            pos_data = opensky.rawdata(
                starttime,
                stoptime,
                Table=PositionData4,
                icao24=icao24s,
                extra_columns=(
                    PositionData4.lat,
                    PositionData4.lon,
                    PositionData4.alt,
                    PositionData4.groundspeed,
                    PositionData4.heading,
                    PositionData4.surface,
                    PositionData4.nic,
                ),
            ).data

            pos_data["timestamp"] = pd.to_datetime(
                pos_data["mintime"], unit="s", errors="coerce"
            ).dt.tz_localize("UTC")

            pos_data = pos_data.rename(
                columns={
                    "lat": "latitude",
                    "lon": "longitude",
                    "surface": "onground",
                    "alt": "altitude",
                    "heading": "track",
                }
            )

            pos_data["altitude"] = pos_data["altitude"].replace(0, pd.NA)
            pos_data["altitude"] = pos_data["altitude"] * 3.28084

            pos_data = pos_data.drop_duplicates(subset=["rawmsg"])

            pos_data = pos_data.dropna(
                subset=[
                    "latitude",
                    "longitude",
                    "groundspeed",
                    "track",
                    "altitude",
                    "nic",
                ],
                how="all",
            )[
                [
                    "timestamp",
                    "icao24",
                    "latitude",
                    "longitude",
                    "groundspeed",
                    "track",
                    "onground",
                    "altitude",
                    "nic",
                ]
            ]

            # Restrict position data to the buffered airport extent.
            pos_data = pos_data.query(
                "longitude > @min_lon and longitude < @max_lon "
                "and latitude > @min_lat and latitude < @max_lat"
            )

            # -------------------------------------------------------------------------
            # Velocity data
            # -------------------------------------------------------------------------

            speed_data = opensky.rawdata(
                starttime,
                stoptime,
                Table=VelocityData4,
                icao24=icao24s,
                extra_columns=(
                    VelocityData4.velocity,
                    VelocityData4.heading,
                    VelocityData4.vertrate,
                ),
            ).data

            speed_data["timestamp"] = pd.to_datetime(
                speed_data["mintime"], unit="s", errors="coerce"
            ).dt.tz_localize("UTC")

            speed_data = speed_data.rename(
                columns={
                    "velocity": "groundspeed",
                    "heading": "track",
                    "vertrate": "vertical_rate",
                }
            )

            speed_data = speed_data.drop_duplicates(subset=["rawmsg"])

            speed_data["groundspeed"] = speed_data["groundspeed"] * 1.94384
            speed_data["vertical_rate"] = speed_data["vertical_rate"] * 196.850394

            diff_binary = (
                speed_data["rawmsg"]
                .str[-20:-6]
                .apply(lambda x: bin(int(x, 16))[2:].zfill(len(x) * 4))
            )

            speed_data["s_diff_alt"] = diff_binary.str[-8]
            speed_data["geominusbaro"] = diff_binary.str[-7:]
            speed_data["geominusbaro"] = (
                speed_data["geominusbaro"].apply(lambda x: int(x, 2)) - 1
            ) * 25

            speed_data.loc[
                speed_data["s_diff_alt"] == "1", "geominusbaro"
            ] = -speed_data.loc[
                speed_data["s_diff_alt"] == "1", "geominusbaro"
            ]

            speed_data = speed_data.dropna(
                subset=["groundspeed", "track", "vertical_rate", "geominusbaro"],
                how="all",
            )[
                [
                    "timestamp",
                    "icao24",
                    "groundspeed",
                    "track",
                    "vertical_rate",
                    "geominusbaro",
                ]
            ]

            # -------------------------------------------------------------------------
            # Merge position and velocity data
            # -------------------------------------------------------------------------

            pos_data = pd.merge_asof(
                pos_data,
                speed_data,
                on="timestamp",
                by="icao24",
                direction="nearest",
                tolerance=pd.Timedelta(seconds=5),
            )

            pos_data["geoaltitude"] = (
                pos_data["altitude"] + pos_data["geominusbaro"]
            )

            pos_data["groundspeed"] = pos_data["groundspeed_x"].combine_first(
                pos_data["groundspeed_y"]
            )

            pos_data["track"] = pos_data["track_x"].combine_first(
                pos_data["track_y"]
            )

            pos_data = pos_data.drop(
                columns=["groundspeed_x", "groundspeed_y", "track_x", "track_y"]
            )

            # -------------------------------------------------------------------------
            # Squawk data
            # -------------------------------------------------------------------------

            rollcall_data = opensky.rawdata(
                starttime,
                stoptime,
                Table=RollcallRepliesData4,
                icao24=icao24s,
                extra_columns=(RollcallRepliesData4.identity,),
            ).data

            rollcall_data = rollcall_data.rename(
                columns={"identity": "squawk"}
            )

            rollcall_data["timestamp"] = pd.to_datetime(
                rollcall_data["mintime"], unit="s", errors="coerce"
            ).dt.tz_localize("UTC")

            rollcall_data = rollcall_data.dropna(subset=["squawk"])[
                ["timestamp", "icao24", "squawk"]
            ]

            pos_data = pd.merge_asof(
                pos_data,
                rollcall_data,
                on="timestamp",
                by="icao24",
                direction="nearest",
                tolerance=pd.Timedelta(minutes=10),
            )

            # -------------------------------------------------------------------------
            # Callsign data
            # -------------------------------------------------------------------------

            identification_data = opensky.rawdata(
                starttime,
                stoptime,
                Table=IdentificationData4,
                icao24=icao24s,
                extra_columns=(IdentificationData4.identity,),
            ).data

            identification_data = identification_data.rename(
                columns={"identity": "callsign"}
            )

            identification_data["callsign"] = (
                identification_data["callsign"]
                .str.replace(" ", "", regex=False)
            )

            identification_data["timestamp"] = pd.to_datetime(
                identification_data["mintime"], unit="s", errors="coerce"
            ).dt.tz_localize("UTC")

            identification_data = identification_data.dropna(
                subset=["callsign"]
            )[
                ["timestamp", "icao24", "callsign"]
            ]

            pos_data = pd.merge_asof(
                pos_data,
                identification_data,
                on="timestamp",
                by="icao24",
                direction="nearest",
                tolerance=pd.Timedelta(minutes=10),
            )

            # -------------------------------------------------------------------------
            # Restrict trajectories to the interval containing valid positions
            # -------------------------------------------------------------------------

            def filter_group(group):
                valid_rows = group[group["latitude"].notna()]

                if valid_rows.empty:
                    return pd.DataFrame()

                start_time = valid_rows["timestamp"].iloc[0]
                end_time = valid_rows["timestamp"].iloc[-1]

                return group[
                    (group["timestamp"] >= start_time)
                    & (group["timestamp"] <= end_time)
                ]

            pos_data = (
                pos_data.groupby("icao24", group_keys=False)
                .apply(filter_group)
                .sort_values(by=["timestamp"])
            )

            t = Traffic(pos_data)
            t.to_parquet(file_path)

        except:
            print(f"{file_path} failed")